In [ ]:
#YEARLY DIVERGENCE ANALYSIS FOR BIOMAKRERS

"""
Sets up the YEARLY DIVERGENCE analysis for OLINK BIOMARKERS.

Biomarker analogue of divergence_yearly.ipynb (pooled composite cognition).
Same question, different measurement:

  "How many years before diagnosis do the biomarker levels of people who will
   develop dementia start to differ from those who will not?"

Run this from the Olink directory (same working directory as
olink_stage2_correlation.ipynb).

DESIGN (nested case-control, per time window) - identical to the cognitive
version, so the two divergence horizons are directly comparable:
- Cases: participants with incident dementia. A case's blood draw contributes
  to the window matching its own lead time (years from draw to diagnosis).
- Controls: participants who never develop dementia, assigned the case's
  'years before index' on their own follow-up.
- Each case is matched to N_CONTROLS controls on age (+/- AGE_CALIPER) and sex.
- In each window: is_case ~ biomarker_z + age_at_test, conditional on the
  matched set. OR is per 1 SD HIGHER biomarker.

The earliest window whose OR lower CI still excludes 1 is the divergence
horizon.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

#minimal plotting size
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.linewidth'] = 0.8

output_dir = 'divergence_yearly_olink'
os.makedirs(output_dir, exist_ok=True)

# ---- config -----------------------------------------------------------
N_CONTROLS   = 10      # controls per case (You et al used 10)
AGE_CALIPER  = 2.0     # years, for age matching
RNG_SEED     = 42
WINDOW_WIDTH = 1       # years per window. Set to 2 if per-year cells are thin
MAX_YEARS    = 25      # furthest lead time considered
MIN_CASES    = 5       # skip windows with fewer cases than this
USE_ADJUSTED = True    # Stage 1 adjusted values (True) or raw NPX (False)
OLINK_INSTANCE = 0     # assessment instance of the blood draw, if the Stage 1
                       # file has no 'instance' column
BIOMARKERS   = None    # None = auto-detect every '<name>_adjusted' column
# -----------------------------------------------------------------------

STAGE1_PATH = 'stage1_adjusted_data.csv'
MASTER_PATH = 'cv_results_all_tests_merged_with_groups.csv'

print(f"Stage 1 biomarkers : {STAGE1_PATH}")
print(f"Survival scaffold  : {MASTER_PATH}")

#load the Stage 1 biomarker file
olink_raw = pd.read_csv(STAGE1_PATH)

if BIOMARKERS is None:
    BIOMARKERS = sorted(c[:-len('_adjusted')] for c in olink_raw.columns
                        if c.endswith('_adjusted'))
suffix = '_adjusted' if USE_ADJUSTED else ''
bm_cols = {bm: f'{bm}{suffix}' for bm in BIOMARKERS}

missing = [c for c in bm_cols.values() if c not in olink_raw.columns]
if missing:
    raise KeyError(f"Columns not in {STAGE1_PATH}: {missing}. "
                   f"Available: {sorted(olink_raw.columns)}")
print(f"Biomarkers         : {', '.join(bm_cols.values())}")

# higher = worse (more pathology). Flip to -1 for any protective marker.
BIOMARKER_DIRECTION = {bm: 1 for bm in BIOMARKERS}

#one biomarker row per person(-instance): Stage 1 repeats the value per test
key = ['eid', 'instance'] if 'instance' in olink_raw.columns else ['eid']
bio = olink_raw.groupby(key, as_index=False)[list(bm_cols.values())].first()
print(f"\nBiomarker rows: {len(bio):,} over {bio['eid'].nunique():,} people "
      f"(keyed on {key})")

#survival / demographic scaffold from the master cognitive file
master = pd.read_csv(MASTER_PATH).rename(columns={'Sex': 'sex', 'Edu': 'edu'})
n_prev = (master['prevalent'] == 1).sum()
master = master[master['prevalent'] == 0].copy()
print(f"Dropped {n_prev:,} prevalent observations from the scaffold")

scaffold = (master.groupby(['eid', 'instance']).agg(
    age_at_test   = ('age_at_test', 'mean'),
    time_to_event = ('time_to_event', 'first'),
    event         = ('event', 'first'),
    sex           = ('sex', 'first'),
    edu           = ('edu', 'first'),
).reset_index())

#merge biomarker draw onto its own visit
if 'instance' in bio.columns:
    visit_df = bio.merge(scaffold, on=['eid', 'instance'], how='inner')
else:
    print(f"No 'instance' column in Stage 1 file - assuming the draw is at "
          f"instance {OLINK_INSTANCE}")
    visit_df = bio.merge(
        scaffold[scaffold['instance'] == OLINK_INSTANCE], on='eid', how='inner')

visit_df = visit_df[visit_df['time_to_event'] > 0].dropna(
    subset=['age_at_test', 'time_to_event', 'event', 'sex']).copy()

if 'sex' not in visit_df.columns or visit_df['sex'].isna().all():
    print("WARNING: no usable 'sex' column - matching on age only.")
    visit_df['sex'] = 0

#standardise each biomarker to SD 1 so the OR is per 1 SD, exactly as the
#cognitive composite was divided by its own SD
for bm, col in bm_cols.items():
    sd = visit_df[col].std(ddof=1)
    visit_df[f'{bm}_z'] = (
        BIOMARKER_DIRECTION[bm] * (visit_df[col] - visit_df[col].mean()) / sd)
    n_ok = visit_df[f'{bm}_z'].notna().sum()
    print(f"{bm.upper():>6}: raw SD = {sd:.4f} (divided out) | "
          f"n with value = {n_ok:,}")

windows = list(range(1, int(np.ceil(MAX_YEARS / WINDOW_WIDTH)) + 1))

print(f"\nDraws = {len(visit_df):,}  |  participants = {visit_df['eid'].nunique():,}")
print(f"Case draws = {int(visit_df['event'].sum()):,}  |  "
      f"control draws = {int((visit_df['event'] == 0).sum()):,}")
print(f"Lead time (cases): median {visit_df.loc[visit_df['event']==1,'time_to_event'].median():.1f} y, "
      f"max {visit_df.loc[visit_df['event']==1,'time_to_event'].max():.1f} y")
print(f"Windows: {len(windows)} x {WINDOW_WIDTH}y, covering 0-{max(windows)*WINDOW_WIDTH}y")

In [ ]:
#Build the matched per-window dataset (Python)
"""
For each case DRAW, its window = floor(time_to_event / WINDOW_WIDTH) + 1
(a draw 3.4 years before diagnosis falls in window 4 at WINDOW_WIDTH = 1,
meaning '3-4 years before'). Each case is matched to N_CONTROLS controls still
at risk at that lead time, within the age caliper and same sex, and the
controls inherit the case's window as their 'years before index'.

Matched sets are built SEPARATELY PER BIOMARKER, using only participants with
a non-missing value for that marker. GFAP and NEFL come off the same panel so
the sets will be near-identical, but this keeps things correct if you add
markers with different missingness.

Result: one long table, one row per (case or matched control) x window x
biomarker, with a binary 'is_case' outcome and the standardised biomarker in
column 'x', ready for per-window conditional logistic regression.
"""
rng = np.random.default_rng(RNG_SEED)

visit_df['window'] = (np.floor(visit_df['time_to_event'] / WINDOW_WIDTH)
                      .astype(int) + 1)
visit_df = visit_df[visit_df['window'].between(1, max(windows))].copy()

matched_frames = []
mid = 0

for bm in BIOMARKERS:
    zcol = f'{bm}_z'
    sub = visit_df.dropna(subset=[zcol]).copy()
    cases    = sub[sub['event'] == 1]
    controls = sub[sub['event'] == 0]
    rows = []

    for w in windows:
        cw = cases[cases['window'] == w]
        # one draw per control person, still at risk at this lead time
        pool = (controls[controls['time_to_event'] >= (w - 1) * WINDOW_WIDTH]
                .sample(frac=1, random_state=RNG_SEED)
                .drop_duplicates('eid'))
        if len(cw) == 0 or len(pool) == 0:
            continue
        pool_age, pool_sex = pool['age_at_test'].values, pool['sex'].values
        pool_idx = pool.index.values

        for _, case in cw.iterrows():
            elig = pool_idx[(np.abs(pool_age - case['age_at_test']) <= AGE_CALIPER)
                            & (pool_sex == case['sex'])]
            if len(elig) == 0:
                continue
            take = rng.choice(elig, size=min(N_CONTROLS, len(elig)), replace=False)
            mid += 1
            rows.append({'match_id': mid, 'biomarker': bm, 'window': w,
                         'is_case': 1, 'x': case[zcol],
                         'age_at_test': case['age_at_test'],
                         'edu': case['edu'], 'eid': case['eid']})
            for ci in take:
                cr = controls.loc[ci]
                rows.append({'match_id': mid, 'biomarker': bm, 'window': w,
                             'is_case': 0, 'x': cr[zcol],
                             'age_at_test': cr['age_at_test'],
                             'edu': cr['edu'], 'eid': cr['eid']})

    bm_matched = pd.DataFrame(rows)
    matched_frames.append(bm_matched)
    print(f"{bm.upper()}: {len(bm_matched):,} rows across "
          f"{bm_matched['window'].nunique()} windows")

matched = pd.concat(matched_frames, ignore_index=True)
matched['match_id'] = matched['match_id'].astype(int)

print(f"\nMatched dataset: {len(matched):,} rows total")
chk = (matched.groupby(['biomarker', 'window'])
       .agg(cases=('is_case', 'sum'), total=('is_case', 'count'),
            people=('eid', 'nunique')))
chk['controls_per_case'] = (chk['total'] - chk['cases']) / chk['cases']
print(chk.round(2).to_string())

thin = chk[chk['cases'] < MIN_CASES]
if len(thin):
    print(f"\n{len(thin)} biomarker-windows below MIN_CASES={MIN_CASES} "
          f"and will be skipped. If most windows are thin, set WINDOW_WIDTH = 2.")

In [ ]:
#Per-window conditional logistic regression in R
"""
-i matched -i windows -i biomarkers_r -i min_cases: imports to R
-o all_results: per-window OR back to Python

For each biomarker b and window w:
  Fit: is_case ~ x + age_at_test + strata(match_id)  (conditional logistic)
  x is the standardised biomarker (higher = worse), so OR > 1 means a higher
  level at that lead time is associated with being a future case - the same
  reading as the cognitive divergence analysis.
"""
%load_ext rpy2.ipython

biomarkers_r = list(BIOMARKERS)
min_cases = int(MIN_CASES)

In [ ]:
%%R -i matched -i windows -i biomarkers_r -i min_cases -o all_results

library(broom)
library(dplyr)
library(survival)

min_cases <- as.integer(min_cases)[1]
all_results <- list()
row_idx <- 1

for (bm in biomarkers_r) {
  cat("\n---", toupper(bm), "---\n")
  for (w in windows) {
    wd <- matched %>% filter(biomarker == bm, window == w)
    n_cases <- sum(wd$is_case)
    if (n_cases < min_cases || nrow(wd) < 2) {
        cat("  window", w, "SKIPPED: (", n_cases, "cases in window)\n")
        next
    }

    fit <- tryCatch(
      clogit(is_case ~ x + age_at_test + strata(match_id), data = wd),
      error = function(e) { cat("  window", w, "ERROR\n"); return(NULL) })
    if (is.null(fit)) next

    tidy_fit <- tidy(fit, conf.int = TRUE, exponentiate = TRUE)
    x_row <- tidy_fit[tidy_fit$term == "x", ]
    if (nrow(x_row) == 0) next

    cat("  window", w, " OR=", round(x_row$estimate, 3),
        " cases=", n_cases, "\n")

    all_results[[row_idx]] <- data.frame(
      biomarker = bm,
      window    = w,
      n_total   = nrow(wd),
      n_cases   = n_cases,
      z_or      = x_row$estimate,
      z_ci_low  = x_row$conf.low,
      z_ci_high = x_row$conf.high,
      z_pval    = x_row$p.value,
      stringsAsFactors = FALSE
    )
    row_idx <- row_idx + 1
  }
}

all_results <- do.call(rbind, all_results)
cat("\nTotal biomarker-windows fit:", nrow(all_results), "\n")

In [ ]:
#Divergence horizon and figures
"""
Divergence horizon: the earliest (largest-w) window whose OR lower CI still
excludes 1. Two versions are reported:

  horizon_max        - furthest significant window, matching the cognitive
                       notebook's definition.
  horizon_contiguous - furthest window reached without a non-significant gap,
                       walking outward from window 1. With the smaller
                       per-window case counts of a single baseline draw, an
                       isolated significant window far out is more likely to
                       be noise, so this is the conservative claim.

Report horizon_max for comparability with the cognitive analysis, but check
the two agree before leaning on the number.
"""
try:
    all_results = pd.DataFrame(all_results)
except Exception:
    all_results = pd.DataFrame()

if len(all_results) == 0 or 'biomarker' not in all_results.columns:
    raise ValueError(
        "No biomarker-windows were fit - all_results came back empty. Every "
        "window fell below MIN_CASES, or every model failed. Check the case "
        "counts printed by the matching cell, then either lower MIN_CASES or "
        "set WINDOW_WIDTH = 2 and re-run from setup.")

all_results = all_results.sort_values(['biomarker', 'window']).reset_index(drop=True)
all_results['years_before'] = all_results['window'] * WINDOW_WIDTH

def horizons(df):
    sig = df[df['z_ci_low'] > 1]
    h_max = int(sig['window'].max()) * WINDOW_WIDTH if len(sig) else 0
    h_con = 0
    for w in sorted(df['window']):
        row = df[df['window'] == w].iloc[0]
        if row['z_ci_low'] > 1:
            h_con = int(w) * WINDOW_WIDTH
        else:
            break
    return h_max, h_con

summary_rows = []
for bm in BIOMARKERS:
    d = all_results[all_results['biomarker'] == bm]
    if len(d) == 0:
        print(f"{bm.upper()}: no windows fit")
        continue
    h_max, h_con = horizons(d)
    summary_rows.append({'biomarker': bm, 'horizon_max': h_max,
                         'horizon_contiguous': h_con,
                         'windows_fit': len(d), 'total_cases': int(d['n_cases'].sum())})
    print(f"\n{'='*62}\n{bm.upper()} divergence horizon: {h_max} years before diagnosis "
          f"(contiguous: {h_con} y)\n{'='*62}")
    print(d[['years_before', 'n_cases', 'z_or', 'z_ci_low', 'z_ci_high', 'z_pval']]
          .round(4).to_string(index=False))

summary = pd.DataFrame(summary_rows)
print("\nSummary:")
print(summary.to_string(index=False))

all_results.to_csv(f'{output_dir}/divergence_results_olink.csv', index=False)
summary.to_csv(f'{output_dir}/divergence_horizons_olink.csv', index=False)

#per-biomarker figure, same style as the cognitive divergence plot
for bm in BIOMARKERS:
    d = all_results[all_results['biomarker'] == bm]
    if len(d) == 0:
        continue
    h_max, _ = horizons(d)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.fill_between(d['years_before'], d['z_ci_low'], d['z_ci_high'],
                    alpha=0.25, color='black')
    ax.plot(d['years_before'], d['z_or'], marker='o', markersize=5,
            color='black', linewidth=2)
    ax.axhline(y=1, color='darkgrey', linestyle='--', linewidth=1)
    if h_max > 0:
        ax.axvline(x=h_max, color='red', linestyle=':', linewidth=1.5)
    ax.invert_xaxis()
    ax.set_xlabel('Years before diagnosis', fontsize=11)
    ax.set_ylabel(f'OR per 1 SD higher {bm.upper()} (case vs matched control)',
                  fontsize=11)
    ax.set_title(f'Yearly divergence ({bm.upper()}): when do future cases diverge?',
                 fontsize=12)
    ax.set_xticks(d['years_before'])
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/divergence_{bm}.png', dpi=800, bbox_inches='tight')
    plt.show()

In [ ]:
#Descriptive trajectory: mean biomarker in cases vs matched controls
"""
The OR curve in odds-ratio space has a plain-language companion: the mean
standardised biomarker level in cases and in their matched controls, by lead
time. Useful as a sanity check (the OR should track the gap) and as the panel
that reads most naturally next to a cognitive trajectory plot.
"""
traj = (matched.groupby(['biomarker', 'window', 'is_case'])['x']
        .agg(['mean', 'sem', 'size']).reset_index())
traj['years_before'] = traj['window'] * WINDOW_WIDTH
traj.to_csv(f'{output_dir}/divergence_trajectories_olink.csv', index=False)

for bm in BIOMARKERS:
    d = traj[traj['biomarker'] == bm]
    if len(d) == 0:
        continue
    fig, ax = plt.subplots(figsize=(8, 5))
    for is_case, label, colour in [(1, 'Future cases', 'black'),
                                   (0, 'Matched controls', 'grey')]:
        s = d[d['is_case'] == is_case].sort_values('years_before')
        ax.plot(s['years_before'], s['mean'], marker='o', markersize=4,
                color=colour, linewidth=1.8, label=label)
        ax.fill_between(s['years_before'], s['mean'] - 1.96 * s['sem'],
                        s['mean'] + 1.96 * s['sem'], alpha=0.2, color=colour)
    ax.axhline(y=0, color='darkgrey', linestyle='--', linewidth=1)
    ax.invert_xaxis()
    ax.set_xlabel('Years before diagnosis', fontsize=11)
    ax.set_ylabel(f'{bm.upper()} (SD units, higher = more pathology)', fontsize=11)
    ax.set_title(f'{bm.upper()} trajectory: cases vs matched controls', fontsize=12)
    ax.legend(frameon=False, fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/trajectory_{bm}.png', dpi=800, bbox_inches='tight')
    plt.show()

In [ ]:
#Overlay: biomarker vs cognitive divergence
"""
The comparison the manuscript actually needs - both divergence curves on one
axis, in the same OR-per-1-SD-worse units, from the same matched design.
Loads the pooled cognitive result if it can find it; skips cleanly if not.
Set COG_RESULTS_PATH manually if the search misses.
"""
COG_CANDIDATES = [
    'divergence_results_pooled.csv', #the data output of 07d_divergence_yearly
]
COG_RESULTS_PATH = next((p for p in COG_CANDIDATES if os.path.exists(p)), None)

if COG_RESULTS_PATH is None:
    print("Cognitive divergence results not found - skipping overlay. "
          f"Tried: {COG_CANDIDATES}")
else:
    print(f"Cognitive results: {COG_RESULTS_PATH}")
    cog = pd.read_csv(COG_RESULTS_PATH).sort_values('window')
    cog['years_before'] = cog['window']   # cognitive notebook used 1y windows

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(cog['years_before'], cog['z_or'], marker='o', markersize=4,
            color='black', linewidth=2, label='Composite cognition')
    ax.fill_between(cog['years_before'], cog['z_ci_low'], cog['z_ci_high'],
                    alpha=0.15, color='black')

    palette = ['#8c564b', '#1f77b4', '#2ca02c', '#d62728', '#9467bd']
    for i, bm in enumerate(BIOMARKERS):
        d = all_results[all_results['biomarker'] == bm]
        if len(d) == 0:
            continue
        c = palette[i % len(palette)]
        ax.plot(d['years_before'], d['z_or'], marker='s', markersize=4,
                color=c, linewidth=2, label=bm.upper())
        ax.fill_between(d['years_before'], d['z_ci_low'], d['z_ci_high'],
                        alpha=0.15, color=c)

    ax.axhline(y=1, color='darkgrey', linestyle='--', linewidth=1)
    ax.invert_xaxis()
    ax.set_xlabel('Years before diagnosis', fontsize=11)
    ax.set_ylabel('OR per 1 SD worse measurement (case vs matched control)',
                  fontsize=11)
    ax.set_title('Divergence horizon: cognition vs blood biomarkers', fontsize=12)
    all_years = sorted(set(cog['years_before']) | set(all_results['years_before']))
    ax.set_xticks(all_years)
    ax.legend(frameon=False, fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/divergence_cognition_vs_biomarkers.png',
                dpi=800, bbox_inches='tight')
    plt.show()